In [ ]:
import json
import random
from openai import OpenAI
from IPython.display import display, JSON
import requests


BASE_URL = "http://localhost:11434/v1/chat/completions"
MODEL = "qwen3:0.6b"   # or llama3 via Groq/Together


def get_current_weather(location):
    # Mock implementation of a weather API call
    return f"The current weather in {location} is sunny with a temperature of {random.randint(60,80)}°F."

#without tools call
def call_llm(BASE_URL, MODEL, user_input):
    
    headers = {
        "Content-Type": "application/json"
    }
    
    message = [
            {"role": "user", "content": user_input}
        ]
    payload = {
            "model": MODEL,
            "messages": message
        }
    
    return requests.post(BASE_URL, headers=headers, json=payload).json()
    

#with tools call
def call_llm_with_tools(BASE_URL, MODEL, message):
    
    headers = {
        "Content-Type": "application/json"
    }
    
    tools = [
            {
                "type": "function",
                "function": {
                    "name": "get_current_weather",
                    "description": "Get the current weather in a given location",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA"
                            }
                        },
                        "required": ["location"]
                    }
                }
            }
        ]
    
    payload = {
        
        "model": MODEL,
        "messages": message,
        "tools": tools
    }
    
    
    return requests.post(BASE_URL, headers=headers, json=payload).json()


In [ ]:

#without tools call
response = call_llm(BASE_URL, MODEL, "Hello")

display(JSON(response,expanded=True))


In [ ]:
#tool call


user_input = "What is the weather like in San Francisco? make sure the tool call is in json format"

message = [
            {"role": "user", "content": user_input}
        ]

# Step 1: First call
response = call_llm_with_tools(BASE_URL, MODEL, message)

display(JSON(response, expanded=True))

msg = response['choices'][0]['message']

# Step 2: Extract tool call properly
tool_calls = msg.get("tool_calls", [])

if tool_calls:
    for tool_call in tool_calls:
        if tool_call["function"]["name"] == "get_current_weather":
            
            args = json.loads(tool_call["function"]["arguments"])
            location = args["location"]

            weather_info = get_current_weather(location)
            print(f"Tool call result: {weather_info}")

            # Step 3: Append assistant message (WITH tool_calls)
            message.append({
                "role": "assistant",
                "content": None,
                "tool_calls": tool_calls
            })

            # Step 4: Append tool response (IMPORTANT FIX)
            message.append({
                "role": "tool",
                "tool_call_id": tool_call["id"],
                "content": weather_info
            })
            
            print("Updated message for second call:")
            print(json.dumps(message, indent=2))

# Step 5: Second call
response2 = call_llm_with_tools(BASE_URL, MODEL, message)

display(JSON(response2, expanded=True))